In [ ]:
!rm -rf /content/PhaseForge
!git clone -b v10 https://github.com/Luke23-45/PhaseForge.git
!git -C PhaseForge log -1 --oneline

In [ ]:
%cd PhaseForge

In [ ]:
%pip install -e ".[dev]"

In [ ]:
import os
# Move to project root
%cd /content/PhaseForge

# Force recreate the virtual environment and reinstall
!rm -rf .venv
!uv venv --python 3.11
!uv pip install -e ".[dev]"

In [ ]:
import os
# Change directory to the repository
os.chdir('/content/PhaseForge')

# Stash local changes and pull
!git stash
!git pull

In [ ]:
!uv sync --extra dev --extra rollout --reinstall-package phaseforge
%env PHASEFORGE_DATA_DIR=/content/data

In [ ]:
!rm -rf /content/PhaseForge/outputs

In [ ]:
from pathlib import Path
import hashlib
import shutil

import h5py
from huggingface_hub import hf_hub_download, get_paths_info

REPO_ID = "amandlek/robomimic"
REPO_TYPE = "dataset"
REVISION = "2aa5bedb20cb20d24b7d857035643cb5deaadbba"

DATA_ROOT = Path("/content/data/raw/robomimic")

TASK_FILES = {
    "lift": "v1.5/lift/ph/low_dim_v15.hdf5",
    "can": "v1.5/can/ph/low_dim_v15.hdf5",
    "square": "v1.5/square/ph/low_dim_v15.hdf5",
    "tool_hang": "v1.5/tool_hang/ph/low_dim_v15.hdf5",
    "transport": "v1.5/transport/ph/low_dim_v15.hdf5",
}

KNOWN_SHA256 = {
    "lift": "2067777cb8b532e9263dd09fd6448c41cc31224bb27be4a3b734010ae13eb540",
}


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def remote_sha256(filename: str) -> str | None:
    info = get_paths_info(
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        paths=filename,
        revision=REVISION,
    )
    if not info:
        return None

    lfs = getattr(info[0], "lfs", None)
    return getattr(lfs, "sha256", None) if lfs is not None else None


for task, filename in TASK_FILES.items():
    destination_dir = DATA_ROOT / task
    destination_dir.mkdir(parents=True, exist_ok=True)
    destination = destination_dir / "low_dim_v15.hdf5"

    expected_sha = KNOWN_SHA256.get(task) or remote_sha256(filename)

    if destination.exists():
        actual_sha = sha256_file(destination)
        if expected_sha is None:
            raise RuntimeError(
                f"No remote SHA-256 metadata available for {task}; "
                f"refusing to trust existing file {destination}."
            )
        if actual_sha != expected_sha:
            raise RuntimeError(
                f"Existing {destination} has SHA-256 {actual_sha}, "
                f"expected {expected_sha}."
            )
        print(f"{task}: existing file verified")
    else:
        cached = hf_hub_download(
            repo_id=REPO_ID,
            repo_type=REPO_TYPE,
            filename=filename,
            revision=REVISION,
        )
        shutil.copy2(cached, destination)

        actual_sha = sha256_file(destination)
        if expected_sha is None:
            raise RuntimeError(
                f"No remote SHA-256 metadata available for {task}; "
                f"downloaded file was not accepted."
            )
        if actual_sha != expected_sha:
            destination.unlink(missing_ok=True)
            raise RuntimeError(
                f"Downloaded {destination} has SHA-256 {actual_sha}, "
                f"expected {expected_sha}."
            )

    with h5py.File(destination, "r") as h5:
        if "data" not in h5:
            raise RuntimeError(f"{destination} is missing the required 'data' group.")
        if "env_args" not in h5["data"].attrs:
            raise RuntimeError(f"{destination} is missing data.attrs['env_args'].")

    print(f"{task}: {destination}")
    print(f"{task}: sha256={sha256_file(destination)}")

print("All five low-dimensional robomimic datasets are verified.")

In [ ]:
import os
os.environ["PHASEFORGE_DATA_DIR"] = "/content/data"

In [ ]:
# Data preparation for the complete five-task state-only protocol.
# Run once after provisioning all five HDF5 files.

import json
import os
from pathlib import Path

import h5py
import numpy as np
import phaseforge
from hydra import compose, initialize_config_module

# Use the external data root used by the provisioning cell.
DATA_ROOT = Path(os.environ.get("PHASEFORGE_DATA_DIR", "/content/data")).resolve()
os.environ["PHASEFORGE_DATA_DIR"] = str(DATA_ROOT)

print(f"PHASEFORGE_DATA_DIR: {DATA_ROOT}")
print(f"phaseforge imported from: {phaseforge.__file__}")

TASKS = ("lift", "can", "square", "tool_hang", "transport")

# Labeler self-test: verifies the installed checkout is current.
from phaseforge.data.robomimic.phase_labeler import RuleBasedPhaseLabeler

_st = np.zeros((40, 9), dtype=np.float32)
_st[:, 7] = np.concatenate([np.full(20, 0.0208), np.full(20, 0.04)])
_st[:, 8] = -_st[:, 7]
_st[:, 0] = np.linspace(0.0, 0.8, 40)

_ph = RuleBasedPhaseLabeler().label({"state": _st})
if len(set(_ph.tolist())) < 3:
    raise RuntimeError(
        "The installed phaseforge checkout is stale: the phase labeler "
        "cannot separate open/closed gripper states. Restart the runtime, "
        "refresh the repository, reinstall phaseforge, and rerun this cell."
    )

print(f"Labeler self-test OK ({len(set(_ph.tolist()))} phases)")

from phaseforge.data.ingestion.state_machine import DataPipelineStateMachine

prepared = {}

with initialize_config_module(
    version_base="1.3",
    config_module="phaseforge.config",
):
    for task in TASKS:
        print(f"\nPreparing task: {task}")

        cfg = compose(
            config_name="main",
            overrides=[
                f"data={task}",
                "models=precision_residual_phaseforge",
                "train=stage1",
                "data.source.auto_download=false",
            ],
        )

        raw_dir = Path(str(cfg.data.source.dir))
        if not raw_dir.is_dir():
            raise RuntimeError(
                f"{task}: source directory is missing: {raw_dir}. "
                "Run the verified five-task provisioning cell first."
            )

        hdf5_files = sorted(raw_dir.glob("*.hdf5"))
        if not hdf5_files:
            raise RuntimeError(f"{task}: no HDF5 file found in {raw_dir}")

        # Validate the raw robomimic container before ingestion.
        for hdf5_path in hdf5_files:
            with h5py.File(hdf5_path, "r") as h5:
                if "data" not in h5:
                    raise RuntimeError(f"{task}: {hdf5_path} has no 'data' group")
                if "env_args" not in h5["data"].attrs:
                    raise RuntimeError(
                        f"{task}: {hdf5_path} is missing data.attrs['env_args']"
                    )

        print(f"Source OK: {raw_dir} ({len(hdf5_files)} HDF5 file(s))")

        pipeline = DataPipelineStateMachine(cfg=cfg)
        dataloaders = pipeline.run()

        train_loader = dataloaders["train"]
        val_loader = dataloaders["val"]

        prepared[task] = {
            "cfg": cfg,
            "pipeline": pipeline,
            "dataloaders": dataloaders,
        }

        print(f"Train samples: {len(train_loader.dataset)}")
        print(f"Val samples:   {len(val_loader.dataset)}")
        print(
            f"Schema: state_dim={cfg.data.state_dim}, "
            f"action_dim={cfg.data.action_dim}, "
            f"phases={cfg.data.phase_labeler.num_phases}"
        )
        print(f"Cache key: {pipeline.config_hash}")

# Preserve compatibility with existing Lift-only notebook method cells.
cfg = prepared["lift"]["cfg"]
pipeline = prepared["lift"]["pipeline"]
dataloaders = prepared["lift"]["dataloaders"]
train_loader = dataloaders["train"]
val_loader = dataloaders["val"]

print("\nAll five task caches are ready.")
print("Lift remains selected as the default notebook task.")
print("Use prepared[task] when running a method for another task.")

In [ ]:
%%bash
set -euo pipefail

cd /content/PhaseForge

VENV=".venv-toolhang"
PYTHON="${VENV}/bin/python"

echo "==> Recreating isolated Python 3.11 environment..."
uv venv --clear --python 3.11 "${VENV}"

echo "==> Installing PhaseForge development dependencies..."
uv pip install \
  --python "${PYTHON}" \
  -e ".[dev]"

echo "==> Pinning tool_hang simulation dependencies..."
uv pip install \
  --python "${PYTHON}" \
  "robosuite==1.5.0" \
  "mujoco==3.2.7"

echo "==> Verifying environment..."
"${PYTHON}" - <<'PY'
import sys
import robosuite
import mujoco

print(f"Python:      {sys.version.split()[0]}")
print(f"Robosuite:   {robosuite.__version__}")
print(f"MuJoCo:      {mujoco.__version__}")

assert sys.version_info[:2] == (3, 11), \
    f"Expected Python 3.11, got {sys.version}"
assert robosuite.__version__ == "1.5.0", \
    f"Expected robosuite 1.5.0, got {robosuite.__version__}"
assert mujoco.__version__ == "3.2.7", \
    f"Expected mujoco 3.2.7, got {mujoco.__version__}"

print("Environment verification: OK")
PY

echo "==> Running tool_hang rollout gate..."
export PHASEFORGE_DATA_DIR="/content/data"
"${VENV}/bin/phaseforge-gates" data=tool_hang eval=rollout


In [ ]:
!uv run phaseforge-gates data=lift eval=rollout
!uv run phaseforge-gates data=can eval=rollout
!uv run phaseforge-gates data=square eval=rollout
!uv run phaseforge-gates data=transport eval=rollout

In [ ]:
!git pull

In [ ]:
!rm -rf /content/PhaseForge/outputs_final

In [ ]:
# 4.1 Proposed (PhaseForge)
!uv run python -m phaseforge.runner \
  --manifest experiments/final_causal_matrix.json \
  --outputs outputs_final \
  --methods precision_residual_phaseforge \
  --tasks Can Square \
  --seeds 45 46 47

In [ ]:
# 4.2 External imitation floor: BC
!uv run python -m phaseforge.runner \
  --manifest experiments/final_causal_matrix.json \
  --outputs outputs_final \
  --methods bc \
  --tasks Can Square \
  --seeds 45 46 47

In [ ]:
# 4.3 Plain-representation control
!uv run python -m phaseforge.runner \
  --manifest experiments/final_causal_matrix.json \
  --outputs outputs_final \
  --methods precision_residual_plain_encoder \
  --tasks Can Square \
  --seeds 45 46 47 \
  --with-dependencies

In [ ]:
# 4.4 Random-router control
!uv run python -m phaseforge.runner \
  --manifest experiments/final_causal_matrix.json \
  --outputs outputs_final \
  --methods precision_residual_phase_random_router \
  --tasks Can Square \
  --seeds 45 46 47 \
  --with-dependencies

In [ ]:
# 4.5 Scratch-expert control
!uv run python -m phaseforge.runner \
  --manifest experiments/final_causal_matrix.json \
  --outputs outputs_final \
  --methods precision_residual_scratch_moe \
  --tasks Can Square \
  --seeds 45 46 47 \
  --with-dependencies

In [ ]:
# 4.6 Two-factor factorial floor
!uv run python -m phaseforge.runner \
  --manifest experiments/final_causal_matrix.json \
  --outputs outputs_final \
  --methods precision_residual_factorial_floor \
  --tasks Can Square \
  --seeds 45 46 47 \
  --with-dependencies

In [ ]:
# 4.7 Softmax top-1 router-package comparison
!uv run python -m phaseforge.runner \
  --manifest experiments/final_causal_matrix.json \
  --outputs outputs_final \
  --methods final_aligned_softmax_top1 \
  --tasks Can Square \
  --seeds 45 46 47 \
  --with-dependencies

In [ ]:
# 4.8 Static-rule integrated comparison
!uv run python -m phaseforge.runner \
  --manifest experiments/final_causal_matrix.json \
  --outputs outputs_final \
  --methods final_aligned_static_rule \
  --tasks Can Square \
  --seeds 45 46 47

In [ ]:
# 4.9 Privileged teacher-forced diagnostic
!uv run python -m phaseforge.runner \
  --manifest experiments/final_causal_matrix.json \
  --outputs outputs_final \
  --methods precision_residual_teacher_forced \
  --tasks Can Square \
  --seeds 45 46 47 \
  --with-dependencies

In [ ]:
# 4.10 Offline oracle diagnostic
!uv run python -m phaseforge.runner \
  --manifest experiments/final_causal_matrix.json \
  --outputs outputs_final \
  --methods precision_residual_oracle \
  --tasks Can Square \
  --seeds 45 46 47 \
  --with-dependencies

In [ ]:
# 5. Router-initialization ablation (Can & Square)
# Controlled ablation behind the paper's T2/A7/A11 assets.
!rm -rf /content/PhaseForge/outputs_ablation
!uv run python -m phaseforge.runner \
  --manifest experiments/router_initialization_ablation_can_square.json \
  --outputs outputs_ablation \
  --seeds 42 43 44 \
  --with-dependencies


In [ ]:
import os
from google.colab import files

# Define an array (list) of directories you want to include in the zip
# Note: Since we cd into /content/PhaseForge, use paths relative to that directory
TARGET_DIRS = ['outputs_final']
ZIP_NAME = '/content/outputs_final_can_square.zip'

# Ensure we are in the repository root
%cd /content/PhaseForge

# Join the array elements into a space-separated string
dirs_to_zip = " ".join(TARGET_DIRS)

print(f"Archiving the following directories: {dirs_to_zip} ...")

# -r: recursive
# -q: quiet
# -x: exclude patterns (quoted to prevent shell expansion)
# We pass {dirs_to_zip} directly to the zip command
!zip -r -q {ZIP_NAME} {dirs_to_zip} -x "*.pt" "*.pth" "*checkpoint*"

# Verify the zip was created and trigger download
if os.path.exists(ZIP_NAME):
    size_mb = os.path.getsize(ZIP_NAME) / (1024 * 1024)
    print(f"Success! Zip created: {ZIP_NAME} ({size_mb:.2f} MB)")
    files.download(ZIP_NAME)
else:
    print("Error: Zip file was not created.")

In [ ]:
!zip -r -q "/content/abalation_final.zip" "outputs_ablation" -x "*.pt" "*.pth" "*checkpoint*"
